# 도메인 평가 에이전트 — 데이터센터 관점

KV cache 최적화 기술 두 건이 **데이터센터** 환경에서 각각 어떤 조건에서 적합하다고
평가받는지 RAG로 근거를 모아 판단하는 에이전트입니다.

- SW: DeepSeek-V2 MLA — 저차원 잠재 압축으로 KV cache 93.3% 감소
- HW: ITME — CXL-Hybrid 계층 메모리로 추론 처리량 1.80배 향상

우열을 판정하지 않고, 관점에 따라 평가가 어떻게 갈리는지를 근거와 함께 기록합니다.
부모 그래프에는 `domain_findings` 키 하나만 반환합니다(`STATE_DESIGN.md` 업데이트 규칙).

흐름은 팀 다이어그램 `outputs/agent-architecture/mermaid/06-domain-evaluation.mmd`를 따릅니다.

```
질문 생성 → RAG 검색 → 적용 조건 검토 → 근거 충분성 판단
                              ├ 부족 & 예산 남음 → 질의 보완 → 재검색
                              └ 충분 또는 한도 도달 → 적합성 분석 → 사실·추론 분리
```

## 왜 데이터센터이고, 왜 RAG인가

**데이터센터 고정 이유.** KV cache 문제의 본질은 연산 병목이 아니라 메모리 병목입니다.
문맥이 길어질수록 저장할 Key-Value가 선형으로 늘어 HBM을 소진하는데, 이 압박이 실제 비용이
되는 곳이 데이터센터입니다. 온디바이스에서는 한 사용자가 자기 메모리를 쓰고 끝나지만,
데이터센터에서는 한 요청이 붙잡은 KV cache가 곧바로 다른 사용자 몫의 HBM을 잠식해
동시 처리 요청 수를 떨어뜨립니다. 또한 HW 진영 기술인 ITME는 CXL 기반 disaggregated memory를
전제하는데 CXL은 랙 스케일 인터커넥트라 온디바이스에는 존재하지 않습니다. 즉 SW 압축과
HW 메모리 확장이 같은 무대에서 비교 가능한 환경은 데이터센터가 사실상 유일합니다.

**RAG를 쓴 이유.** ITME 논문은 2026-06 공개 자료로 GPT 학습 데이터 절단 이후일 가능성이 높아
파라미터 지식에만 의존하면 수치를 지어낼 위험이 큽니다. 또 최종 보고서는 REFERENCE 장에
실제 사용한 자료만 기재해야 하므로 주장마다 문서·페이지가 연결되어야 합니다.
마지막으로, 그냥 물으면 논문 주장을 되풀이하는 긍정 편향이 생기므로 검색으로 제약·한계
근거까지 확보해 "근거를 찾지 못했다"는 판단 보류가 가능하게 했습니다.

자세한 근거는 `docs/DOMAIN_AGENT.md` 참조.

In [1]:
# 노트북이 notebooks/ 안에 있으므로 레포 루트를 import 경로에 추가한다.
# state.py, agents/, prompts/ 가 루트에 있기 때문이다.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# OPENAI_API_KEY 를 읽는다. 평가 LLM은 GPT를 사용한다.
from dotenv import load_dotenv

load_dotenv(ROOT / ".env", override=True)
import os

assert os.environ.get("OPENAI_API_KEY"), ".env 에 OPENAI_API_KEY 를 설정하세요"
print("루트:", ROOT)

루트: /Users/sanlee/Desktop/SKALA/코딩파일/Ai-service/Capstone_Ai_RAG


## 1. 데모용 검색 인덱스

에이전트는 인덱스를 직접 만들지 않고 `Retriever` 프로토콜을 만족하는 검색 도구를 주입받습니다.
인덱스 구축은 팀 공유 RAG 준비 단계가 담당하므로, 아래는 **공유 인덱스가 준비되기 전에
동작을 확인하기 위한 데모**입니다. 공유 도구가 나오면 이 부분만 교체하면 됩니다.

In [2]:
# 선정 논문 2편을 내려받는다(이미 있으면 건너뛴다).
# 용량이 커서 git에는 올리지 않고 .gitignore 로 제외한다.
import urllib.request

PAPERS = {
    "deepseek-v2.pdf": "https://arxiv.org/pdf/2405.04434",
    "itme-cxl.pdf": "https://arxiv.org/pdf/2606.12556",
}
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

for name, url in PAPERS.items():
    target = DATA_DIR / name
    if target.exists():
        print(f"이미 있음: {name} ({target.stat().st_size // 1024}KB)")
        continue
    urllib.request.urlretrieve(url, target)
    print(f"내려받음: {name} ({target.stat().st_size // 1024}KB)")

이미 있음: deepseek-v2.pdf (1549KB)
이미 있음: itme-cxl.pdf (2812KB)


In [3]:
# corpus_manifest: 팀 state.py 계약에 정의된 문서 목록이다.
# indexed_pages 는 1-based 이며, 모든 문서의 선정 페이지 합이 200을 넘으면
# initial_state() 가 예외를 던진다(과제의 200페이지 제한을 코드로 강제).
import pdfplumber


def manifest_entry(doc_id: str, title: str, url: str, filename: str, org: str, published: str):
    path = DATA_DIR / filename
    with pdfplumber.open(str(path)) as pdf:
        total = len(pdf.pages)
    return {
        "document_id": doc_id,
        "title": title,
        "source_url": url,
        "local_path": str(path),
        "total_pages": total,
        "indexed_pages": list(range(1, total + 1)),  # 두 편 합계가 200p 미만이라 전체 사용
        "author_or_organization": org,
        "published_date": published,
    }


corpus_manifest = {
    "documents": [
        manifest_entry(
            "sw-deepseek-v2",
            "DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts Language Model",
            "https://arxiv.org/pdf/2405.04434",
            "deepseek-v2.pdf",
            "DeepSeek-AI",
            "2024-06",
        ),
        manifest_entry(
            "hw-itme-cxl",
            "ITME: Inference Tiered Memory Expansion with Disaggregated CXL-Hybrid Memories",
            "https://arxiv.org/pdf/2606.12556",
            "itme-cxl.pdf",
            "Jang, H. et al.",
            "2026-06",
        ),
    ],
    # 과제 요구사항: 오픈소스 임베딩 필수. 선정 사유는 docs/DOMAIN_AGENT.md 의 비교표 참조.
    "embedding_model": "BAAI/bge-m3",
    "embedding_selection_reason": (
        "한국어 질의로 영어 논문을 찾아야 해 교차언어 검색이 필수였고, "
        "8192 토큰 입력이라 표·수식이 포함된 긴 청크를 자르지 않는다. MIT 라이선스."
    ),
    "vector_index_uri": "memory://faiss-demo",
    "keyword_index_uri": "memory://bm25-demo",
}

total_pages = sum(len(d["indexed_pages"]) for d in corpus_manifest["documents"])
for d in corpus_manifest["documents"]:
    print(f"{d['document_id']}: {len(d['indexed_pages'])}p / 전체 {d['total_pages']}p")
print(f"선정 페이지 합계: {total_pages}p (제한 200p)")

sw-deepseek-v2: 52p / 전체 52p
hw-itme-cxl: 13p / 전체 13p
선정 페이지 합계: 65p (제한 200p)


In [4]:
# 데모 인덱스: 벡터(FAISS) + 키워드(BM25) 하이브리드.
# 하이브리드로 짠 이유는 질의 유형이 두 가지이기 때문이다.
#  - "메모리 압박을 어떻게 완화하나" 같은 의미 질의 -> 벡터 검색이 강함
#  - "MLA", "CXL", "93.3%" 같은 고유명사/수치 -> 키워드 검색이 정확히 집음
# 도메인 평가는 두 유형을 모두 쓰므로 한쪽만으로는 근거를 놓친다.
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from agents.domain_agent import SearchHit


def load_selected_pages(manifest: dict) -> list:
    """manifest 에 선정된 페이지만 남긴다. 200p 제한을 지키는 실제 지점.

    PDFPlumberLoader 의 metadata['page'] 는 0-based 라 계약(1-based)에 맞춰 +1 한다.
    """
    pages = []
    for doc in manifest["documents"]:
        wanted = set(doc["indexed_pages"])
        for page in PDFPlumberLoader(doc["local_path"]).load():
            page_no = int(page.metadata.get("page", 0)) + 1
            if page_no not in wanted:
                continue
            # 출처 메타데이터를 여기서 붙이지 않으면 보고서 REFERENCE 단계에서 복원할 수 없다.
            page.metadata.update(
                {
                    "document_id": doc["document_id"],
                    "title": doc["title"],
                    "source_url": doc["source_url"],
                    "page_number": page_no,
                    "author_or_organization": doc["author_or_organization"],
                    "published_date": doc["published_date"],
                }
            )
            pages.append(page)
    return pages


class DemoHybridRetriever:
    """agents.domain_agent.Retriever 프로토콜 구현. 팀 공유 검색 도구로 교체 대상."""

    def __init__(self, manifest: dict):
        pages = load_selected_pages(manifest)
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        self.chunks = splitter.split_documents(pages)
        embeddings = HuggingFaceEmbeddings(model_name=manifest["embedding_model"])
        self.vector = FAISS.from_documents(self.chunks, embeddings)
        self.keyword = BM25Retriever.from_documents(self.chunks)

    def search(self, query: str, k: int = 4) -> list[SearchHit]:
        docs = []
        # 한쪽 인덱스가 실패해도 나머지로 진행한다. 검색 실패가 그래프를 중단시키면 안 된다.
        try:
            docs.extend(self.vector.similarity_search(query, k=k))
        except Exception as exc:
            print("벡터 검색 실패:", exc)
        try:
            self.keyword.k = k
            docs.extend(self.keyword.invoke(query))
        except Exception as exc:
            print("키워드 검색 실패:", exc)

        hits, seen = [], set()
        for doc in docs:
            meta = doc.metadata
            excerpt = doc.page_content.strip()
            key = (meta.get("document_id"), meta.get("page_number"), excerpt[:120])
            if key in seen:  # 두 인덱스가 같은 청크를 반환하는 경우 제거
                continue
            seen.add(key)
            hits.append(
                SearchHit(
                    title=meta.get("title", "unknown"),
                    url=meta.get("source_url", ""),
                    excerpt=excerpt[:800],
                    source_type="paper",
                    author_or_organization=meta.get("author_or_organization", "unknown"),
                    document_id=meta.get("document_id"),
                    page=meta.get("page_number"),
                    published_date=meta.get("published_date"),
                )
            )
        return hits[: k * 2]


# BGE-M3 최초 실행 시 모델을 내려받으므로 수 분 걸릴 수 있다.
retriever = DemoHybridRetriever(corpus_manifest)
print(f"색인 청크 수: {len(retriever.chunks)}")

/var/folders/z0/m8vvx8mn5bvdq70sfbkm3wy40000gn/T/ipykernel_26640/3547659895.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PDFPlumberLoader
/Users/sanlee/Desktop/SKALA/코딩파일/Ai-service/Capstone_Ai_RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 54056.72it/s]

색인 청크 수: 219


In [5]:
# 검색이 제대로 동작하는지, 그리고 출처 메타데이터가 붙는지 먼저 확인한다.
# 한국어 질의로 영어 논문을 찾는 교차언어 검색이 BGE-M3 를 고른 이유였다.
for hit in retriever.search("MLA가 KV 캐시를 얼마나 줄이는가", k=2)[:3]:
    print(f"[{hit.document_id} p.{hit.page}] {hit.excerpt[:110]}...\n")

[sw-deepseek-v2 p.4] activatedforeachtoken,andsupportsacontextlengthof128Ktokens.
WeoptimizetheattentionmodulesandFeed-ForwardNetwo...

[sw-deepseek-v2 p.31] themon1.33Ttokens. TwolargeMoEmodelscompriseabout250Btotalparameters,andwe
trainthemon420Btokens. Also,twosmal...

[hw-itme-cxl p.2] Table1:LLMDataCharacteristicsandTargetTiering
GPU Memory (T1) GPU Memory (T1)
DataType Predict. Perf. TargetTi...



## 2. 입력 State 구성

부모 그래프에서 도메인 노드는 `request`, `corpus_manifest`, 그리고 선행 단계인
기술 조사 에이전트의 `technical_findings` 를 읽습니다.
여기서는 기술 조사 결과를 최소 형태로 모사해 넣습니다(팀원이 담당하는 부분).

In [6]:
from state import initial_state

request = {
    "sw": {
        "name": "DeepSeek-V2 MLA (Multi-head Latent Attention)",
        "selection_reason": "저차원 잠재 압축으로 KV cache 93.3% 감소. 사후 압축이 아닌 아키텍처 개선 계열 대표",
        "seed_urls": ["https://arxiv.org/pdf/2405.04434"],
    },
    "hw": {
        "name": "ITME (Inference Tiered Memory Expansion, CXL-Hybrid)",
        "selection_reason": "CXL-Hybrid 계층 메모리로 대규모 LLM 추론 처리량 1.80배 향상. 정확도 손실 없는 용량 확장 계열 대표",
        "seed_urls": ["https://arxiv.org/pdf/2606.12556"],
    },
    "domains": ["데이터센터"],
    "as_of_date": "2026-09-21",
    "language": "ko",
    "max_search_rounds": 2,  # 최초 검색 포함. 근거가 부족하면 1회 더 질의를 바꿔 재검색
    "max_revision_rounds": 1,
}

# initial_state 가 200페이지 제한·날짜 형식·문서 ID 중복 등을 검증한다.
state = initial_state("domain-demo-001", request, corpus_manifest)

# 선행 기술 조사 에이전트의 산출물 모사. 실제 실행에서는 technical 노드가 채운다.
state["technical_findings"] = {
    "claims": [
        {
            "claim_id": "technical:claim:001",
            "technology_ids": ["sw"],
            "topic": "원리",
            "statement": "MLA는 KV를 저차원 잠재 벡터로 압축해 캐시 크기를 93.3% 줄인다고 보고됨",
            "basis": "direct_evidence",
            "evidence_ids": ["technical:ev:001"],
            "conditions": ["DeepSeek-V2 236B MoE 기준"],
            "uncertainty": "자체 보고 수치",
        },
        {
            "claim_id": "technical:claim:002",
            "technology_ids": ["hw"],
            "topic": "원리",
            "statement": "ITME는 CXL-Hybrid 계층 메모리로 KV cache를 HBM 밖으로 확장해 처리량 1.80배를 보고",
            "basis": "direct_evidence",
            "evidence_ids": ["technical:ev:002"],
            "conditions": ["논문 실험 구성 기준"],
            "uncertainty": "실제 운영 환경 검증 여부 불명",
        },
    ],
    "evidence": [],
    "completion": {
        "status": "complete",
        "search_rounds_used": 1,
        "revision_rounds_used": 0,
        "gaps": [],
        "errors": [],
    },
    "trl_estimates": [
        {
            "technology_id": "sw",
            "level": 7,
            "rationale": "상용 모델에 적용되어 공개 서비스 중",
            "evidence_ids": [],
            "caveat": "공개 정보 기반 추정",
        },
        {
            "technology_id": "hw",
            "level": 4,
            "rationale": "연구 프로토타입 수준 시연",
            "evidence_ids": [],
            "caveat": "공개 정보 기반 추정",
        },
    ],
    "comparison_caveats": ["두 기술의 측정 환경이 달라 수치 직접 비교 불가"],
}
print("run_id:", state["run_id"], "/ 도메인:", state["request"]["domains"])

run_id: domain-demo-001 / 도메인: ['데이터센터']


## 3. 에이전트 실행

`build_domain_agent` 가 반환하는 함수를 그대로 `graph_wiring.build_graph(domain=...)` 에
주입하면 부모 그래프에 붙습니다. 여기서는 단독으로 호출해 동작을 확인합니다.

In [7]:
from agents.domain_agent import build_domain_agent, make_deps

# LLM(GPT)과 검색 도구는 State 밖의 런타임 의존성으로 주입한다.
# State 에는 JSON 직렬화 가능한 값만 넣는다는 팀 규칙(STATE_DESIGN.md) 때문이다.
deps = make_deps(retriever=retriever, model="gpt-4o-mini")
domain_node = build_domain_agent(deps)

result = domain_node(state)

# 부모 State 에 돌려주는 키가 domain_findings 하나뿐인지 확인한다.
# 병렬 분기에서 다른 관점 에이전트와 키가 충돌하지 않게 하는 핵심 계약이다.
print("반환 키:", list(result.keys()))
findings = result["domain_findings"]

반환 키: ['domain_findings']


In [8]:
# completion: 이 에이전트가 어디까지 해냈는지를 종합 단계가 읽는 부분.
#  complete = 공백 없이 완료 / partial = 결과는 냈으나 못 채운 축 있음 / failed = 판단 생성 실패
c = findings["completion"]
print(f"status={c['status']}  검색 라운드={c['search_rounds_used']}")
print(f"근거 {len(findings['evidence'])}건 / 주장 {len(findings['claims'])}건 / 판정 {len(findings['fits'])}건")

if c["gaps"]:
    print("\n못 채운 부분(gaps) — 보고서 한계점 장에 반영된다")
    for g in c["gaps"]:
        print(" -", g)

status=partial  검색 라운드=2
근거 27건 / 주장 7건 / 판정 2건

못 채운 부분(gaps) — 보고서 한계점 장에 반영된다
 - 전력·발열
 - 인프라 도입 비용과 운영 부담


In [9]:
# fits: 종합 에이전트가 관점 간 상충을 기계적으로 찾을 수 있도록 열거형으로 고정한 판정.
#  suitable / conditional / unsuitable / unknown 중 하나만 나온다.
for f in findings["fits"]:
    print(f"[{f['technology_id']}] {f['domain']} 적합성: {f['assessment']}")
    print(f"  근거 주장: {f['claim_ids']}")
    for lim in f["limitations"][:3]:
        print(f"  제약: {lim}")
    print()

[sw] 데이터센터 적합성: suitable
  근거 주장: ['domain:claim:001', 'domain:claim:003']

[hw] 데이터센터 적합성: conditional
  근거 주장: ['domain:claim:004', 'domain:claim:006', 'domain:claim:007']
  제약: 신규 하드웨어가 필요할 수 있음
  제약: 전력 예산이 제한적일 경우 성능 저하 우려



In [10]:
# claims: 판정을 뒷받침하는 주장. basis 로 사실과 추론을 분리한다.
#  direct_evidence = 근거가 직접 뒷받침 / inference = 근거에서 유추 / unknown = 판단 보류
# 검증 단계에서 존재하지 않는 근거를 인용하면 자동으로 강등된다.
for cl in findings["claims"]:
    print(f"[{cl['claim_id']}] ({cl['basis']}) {' '.join(cl['technology_ids'])}")
    print(f"  {cl['statement']}")
    print(f"  근거: {cl['evidence_ids']}")
    if cl["conditions"]:
        print(f"  전제: {cl['conditions']}")
    print(f"  한계: {cl['uncertainty']}\n")

[domain:claim:001] (direct_evidence) sw
  DeepSeek-V2는 KV 캐시 크기를 93.3% 줄여 HBM 용량 압박을 완화할 수 있다.
  근거: ['domain:ev:001', 'domain:ev:004']
  전제: ['멀티테넌시 환경에서 동시 요청 수가 많을 때', '기존 서버에 즉시 적용 가능할 때']
  한계: 

[domain:claim:002] (inference) sw
  DeepSeek-V2는 Multi-Head Latent Attention을 통해 지연시간을 줄일 수 있지만, 데이터센터의 표준 환경과는 다를 수 있다.
  근거: ['domain:ev:003', 'domain:ev:012']
  전제: ['지연시간이 중요한 서비스 SLA를 만족해야 할 때', 'FP8 정밀도를 지원하는 환경에서']
  한계: DeepSeek-V2의 최적화가 데이터센터의 일반적인 환경과 다를 수 있음.

[domain:claim:003] (inference) sw
  DeepSeek-V2의 KV 캐시 압축은 품질 저하를 유발할 수 있어 손실 허용 범위를 확인해야 한다.
  근거: ['domain:ev:006', 'domain:ev:018']
  전제: ['압축률을 높일 때 품질 저하가 우려되는 경우']
  한계: 근거에서 유추한 판단으로 적용 범위가 제한됨

[domain:claim:004] (direct_evidence) hw
  ITME는 CXL-Hybrid 메모리로 KV 캐시를 HBM 밖으로 확장하여 처리량을 1.80배 증가시킬 수 있다.
  근거: ['domain:ev:007', 'domain:ev:023']
  전제: ['고속 데이터 전송이 필요한 환경에서', '기존 하드웨어와의 호환성이 있을 때']
  한계: 

[domain:claim:005] (inference) hw
  ITME는 데이터 전송 지연을 최소화하기 위해 레이어별 프리패칭을 사용하지만, 데이터센터의 환경에 따라 성능이 달라질 수 있다.
  근거: ['d

In [11]:
# evidence: 보고서 REFERENCE 장을 만들 수 있는 형태인지 확인한다.
# 문서 ID·페이지·URL 이 모두 붙어 있어야 인용 추적이 가능하다.
for ev in findings["evidence"][:3]:
    print(f"[{ev['evidence_id']}] ({ev['source_type']}) {ev['title'][:60]}")
    print(f"  {ev['author_or_organization']} ({ev['published_date']}) p.{ev['page']}")
    print(f"  {ev['url']}")
    print(f"  발췌: {ev['excerpt'][:100]}...\n")

[domain:ev:001] (paper) DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-
  DeepSeek-AI (2024-06) p.21
  https://arxiv.org/pdf/2405.04434
  발췌: stronger performance, and meanwhile saves 42.5% of training costs, reduces the KV cache
by 93.3%, an...

[domain:ev:002] (paper) DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-
  DeepSeek-AI (2024-06) p.6
  https://arxiv.org/pdf/2405.04434
  발췌: and DeepSeekMoE in this section. For other tiny details (e.g., layer normalization and the
activatio...

[domain:ev:003] (paper) DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-
  DeepSeek-AI (2024-06) p.4
  https://arxiv.org/pdf/2405.04434
  발췌: activatedforeachtoken,andsupportsacontextlengthof128Ktokens.
WeoptimizetheattentionmodulesandFeed-Fo...



## 4. 종합 평가 에이전트로 넘기는 데이터

평가 결과를 긴 산문으로 넘기면 종합 단계에서 내용이 희석됩니다. 여러 관점의 텍스트가
한 프롬프트에 들어가면 어느 주장이 어느 근거에 붙어 있었는지 잃어버리고
"대체로 유망하다" 같은 뭉뚱그린 종합이 나옵니다. 그래서 산문 대신 구조화 레코드로 넘기며,
다음 세 가지를 코드에서 강제합니다.

- 주장 1건 = 한 문장(240자 제한). 길면 여러 논점이 섞여 종합 단계에서 분해가 불가능합니다.
- 판단은 자유 서술이 아니라 열거형. 종합은 값만 보고 상충 지점을 기계적으로 찾습니다.
- 근거 본문은 반복해 싣지 않고 ID로 참조. 원문은 `evidence` 에 한 번만 보관합니다.

In [12]:
# 실제로 계약이 지켜졌는지 검사한다. TypedDict 는 런타임 검증을 하지 않으므로 직접 확인한다.
import json

ev_ids = {e["evidence_id"] for e in findings["evidence"]}
claims_by_id = {c["claim_id"]: c for c in findings["claims"]}
problems = []

for cl in findings["claims"]:
    if len(cl["statement"]) > 240:
        problems.append(f"{cl['claim_id']}: 문장 길이 초과")
    if any(e not in ev_ids for e in cl["evidence_ids"]):
        problems.append(f"{cl['claim_id']}: 존재하지 않는 근거 참조")
    if cl["basis"] == "direct_evidence" and not cl["evidence_ids"]:
        problems.append(f"{cl['claim_id']}: 근거 없는 사실 주장")
    if cl["basis"] == "inference" and not cl["conditions"]:
        problems.append(f"{cl['claim_id']}: 추론인데 전제가 없음")

for f in findings["fits"]:
    for cid in f["claim_ids"]:
        if cid not in claims_by_id:
            problems.append(f"{f['technology_id']}: 존재하지 않는 주장 참조")
            continue
        # ID 유효성만 보면 sw 판정이 hw 주장을 가리켜도 통과한다. 기술 일치까지 확인한다.
        if f["technology_id"] not in claims_by_id[cid]["technology_ids"]:
            problems.append(f"{f['technology_id']} 판정이 다른 기술 주장({cid})을 참조")

# State 에는 JSON 직렬화 가능한 값만 들어가야 한다(체크포인터 저장 경계).
payload = json.dumps(findings, ensure_ascii=False)

print("계약 검증:", "통과" if not problems else problems)
print(f"직렬화 크기: {len(payload):,} bytes")
print(f"종합 단계가 읽는 핵심: fits {len(findings['fits'])}건 + claims {len(findings['claims'])}건")

계약 검증: 통과
직렬화 크기: 33,696 bytes
종합 단계가 읽는 핵심: fits 2건 + claims 7건


## 정리

- 이 에이전트는 인덱스를 소유하지 않고 `Retriever` 프로토콜만 요구하므로,
  팀 공유 검색 도구가 완성되면 `make_deps(retriever=...)` 인자만 바꾸면 됩니다.
- 부모 그래프에는 `domain_findings` 한 키만 반환해 병렬 분기에서 키 충돌이 없습니다.
- 근거가 부족하면 임의로 메우지 않고 `unknown` 과 `gaps` 로 남기며,
  이는 보고서의 한계점 장으로 이어집니다.